# E-commerce Sales Data Analysis Using Python
## Module 02 – Week 2 | Data Science Virtual Internship

This project performs exploratory data analysis (EDA) on the **UCI Online Retail** real-world transaction dataset.

The dataset contains transactions from a UK-based online retailer between 1 December 2010 and 9 December 2011. Each row represents an item on an invoice. The company mainly sells gifts and many customers are wholesalers.

**Project questions**
- How do sales change over time?
- Which products and countries generate the most sales revenue?
- How many orders and customers are represented?
- What patterns can be found after cleaning cancellations, invalid quantities and prices?

**Tools:** Python, NumPy, Pandas, Matplotlib and Google Colab.

> Dataset source: Chen, D. (2015). Online Retail. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

## 1. Import libraries

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries imported successfully.")

## 2. Download and load the real-world dataset

In [ ]:
# Download the official UCI dataset archive.
data_dir = Path("ecommerce_data")
data_dir.mkdir(exist_ok=True)

zip_path = data_dir / "online_retail.zip"
dataset_url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

try:
    if not zip_path.exists():
        urllib.request.urlretrieve(dataset_url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as archive:
        excel_files = [name for name in archive.namelist()
                       if name.lower().endswith(".xlsx")]
        if not excel_files:
            raise FileNotFoundError("No Excel dataset found inside the UCI archive.")

        excel_name = excel_files[0]
        archive.extract(excel_name, data_dir)
        excel_path = data_dir / excel_name

    print("Dataset downloaded from the UCI Machine Learning Repository.")
    df = pd.read_excel(excel_path, engine="openpyxl")

except Exception as error:
    print("Automatic download failed:", error)
    print("Download Online Retail.xlsx from the UCI page and upload it here.")
    from google.colab import files
    uploaded = files.upload()
    excel_name = next(name for name in uploaded if name.lower().endswith(".xlsx"))
    df = pd.read_excel(excel_name, engine="openpyxl")

display(df.head())

## 3. Understand the dataset

In [ ]:
print("Dataset shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

## 4. Standardize column names and inspect missing values

In [ ]:
# Convert column labels to lowercase snake_case.
df.columns = (
    df.columns.str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

print("Missing values in each column:")
missing_summary = df.isna().sum().to_frame("missing_values")
missing_summary["missing_percent"] = (df.isna().mean() * 100).round(2)
display(missing_summary)

print("Exact duplicate rows:", df.duplicated().sum())

## 5. Clean the dataset

In [ ]:
# Work on a copy so the original imported data remains unchanged.
clean_df = df.copy()

# Convert date and numeric columns to appropriate data types.
clean_df["invoice_date"] = pd.to_datetime(
    clean_df["invoice_date"], errors="coerce", dayfirst=True
)
clean_df["quantity"] = pd.to_numeric(clean_df["quantity"], errors="coerce")
clean_df["unitprice"] = pd.to_numeric(clean_df["unitprice"], errors="coerce")

# Remove exact duplicate rows.
clean_df = clean_df.drop_duplicates().copy()

# Remove rows that cannot be used for line-level sales analysis.
required_columns = ["invoice_no", "stockcode", "description",
                    "invoice_date", "quantity", "unitprice"]
clean_df = clean_df.dropna(subset=required_columns).copy()

# Trim product descriptions.
clean_df["description"] = clean_df["description"].astype(str).str.strip()

print("Rows after duplicate and essential-field cleaning:", len(clean_df))
print("Missing values remaining in required fields:",
      int(clean_df[required_columns].isna().sum().sum()))
display(clean_df.head())

## 6. Identify cancellations, returns and invalid sales lines

In [ ]:
# UCI notes that invoice numbers beginning with C indicate cancellations.
clean_df["is_cancellation"] = (
    clean_df["invoice_no"].astype(str).str.upper().str.startswith("C")
)

print("Cancellation rows:", int(clean_df["is_cancellation"].sum()))
print("Rows with zero or negative quantity:",
      int((clean_df["quantity"] <= 0).sum()))
print("Rows with zero or negative unit price:",
      int((clean_df["unitprice"] <= 0).sum()))

# For this sales analysis, retain completed positive-value sales lines only.
sales_df = clean_df[
    (~clean_df["is_cancellation"]) &
    (clean_df["quantity"] > 0) &
    (clean_df["unitprice"] > 0)
].copy()

# Revenue is calculated in British pounds (GBP), the dataset's stated currency.
sales_df["revenue"] = sales_df["quantity"] * sales_df["unitprice"]

print("Valid sales rows:", len(sales_df))
display(sales_df[["invoice_no", "description", "quantity",
                  "unitprice", "revenue", "country"]].head())

## 7. Create useful date features

In [ ]:
sales_df["year"] = sales_df["invoice_date"].dt.year
sales_df["month"] = sales_df["invoice_date"].dt.to_period("M").astype(str)
sales_df["day_name"] = sales_df["invoice_date"].dt.day_name()

display(sales_df[["invoice_date", "year", "month", "day_name"]].head())

## 8. Calculate key business metrics

In [ ]:
total_revenue = sales_df["revenue"].sum()
total_orders = sales_df["invoice_no"].nunique()
total_products = sales_df["stockcode"].nunique()
total_countries = sales_df["country"].nunique()
identified_customers = sales_df["customer_id"].nunique()

print(f"Total sales revenue: £{total_revenue:,.2f}")
print(f"Number of completed orders: {total_orders:,}")
print(f"Distinct products sold: {total_products:,}")
print(f"Countries represented: {total_countries:,}")
print(f"Identified customers: {identified_customers:,}")
print(f"Average order value: £{total_revenue / total_orders:,.2f}")

## 9. Analyze monthly sales revenue

In [ ]:
monthly_sales = (
    sales_df.groupby("month")["revenue"]
    .sum()
    .sort_index()
)

display(monthly_sales.to_frame("revenue_gbp").round(2))

plt.figure()
plt.plot(monthly_sales.index, monthly_sales.values, marker="o")
plt.title("Monthly E-commerce Sales Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (£ GBP)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Find the top 10 products by revenue

In [ ]:
top_products = (
    sales_df.groupby("description")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

display(top_products.to_frame("revenue_gbp").round(2))

plt.figure()
top_products.sort_values().plot(kind="barh", edgecolor="black")
plt.title("Top 10 Products by Sales Revenue")
plt.xlabel("Revenue (£ GBP)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

## 11. Compare sales revenue by country

In [ ]:
country_sales = (
    sales_df.groupby("country")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

top_countries = country_sales.head(10)
display(top_countries.to_frame("revenue_gbp").round(2))

plt.figure()
top_countries.sort_values().plot(kind="barh", edgecolor="black")
plt.title("Top 10 Countries by Sales Revenue")
plt.xlabel("Revenue (£ GBP)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## 12. Analyze order and customer activity

In [ ]:
orders_per_customer = (
    sales_df.dropna(subset=["customer_id"])
    .groupby("customer_id")["invoice_no"]
    .nunique()
)

customer_revenue = (
    sales_df.dropna(subset=["customer_id"])
    .groupby("customer_id")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("Average orders per identified customer:",
      round(orders_per_customer.mean(), 2))
print("\nTop 10 customers by sales revenue:")
display(customer_revenue.head(10).to_frame("revenue_gbp").round(2))

plt.figure()
plt.hist(orders_per_customer, bins=30, edgecolor="black")
plt.title("Distribution of Orders per Customer")
plt.xlabel("Number of orders")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

## 13. Group and aggregate sales data

In [ ]:
country_summary = sales_df.groupby("country").agg(
    total_revenue=("revenue", "sum"),
    total_quantity=("quantity", "sum"),
    number_of_orders=("invoice_no", "nunique"),
    average_unit_price=("unitprice", "mean")
).sort_values("total_revenue", ascending=False).round(2)

display(country_summary.head(10))

monthly_summary = sales_df.groupby("month").agg(
    total_revenue=("revenue", "sum"),
    total_quantity=("quantity", "sum"),
    number_of_orders=("invoice_no", "nunique")
).sort_index().round(2)

display(monthly_summary)

## 14. Explore the relationship between quantity and revenue

In [ ]:
# Aggregate at invoice level to compare order size and order revenue.
order_summary = sales_df.groupby("invoice_no").agg(
    order_quantity=("quantity", "sum"),
    order_revenue=("revenue", "sum")
)

display(order_summary.describe().round(2))

plt.figure()
plt.scatter(order_summary["order_quantity"],
            order_summary["order_revenue"], alpha=0.35)
plt.title("Order Quantity vs Order Revenue")
plt.xlabel("Total items in order")
plt.ylabel("Order revenue (£ GBP)")
plt.tight_layout()
plt.show()

print("Correlation between order quantity and order revenue:",
      round(order_summary["order_quantity"].corr(
          order_summary["order_revenue"]), 3))

## 15. Filter and sort records

In [ ]:
# Example: display the 10 highest-value individual sales lines.
highest_value_lines = sales_df.sort_values(
    "revenue", ascending=False
)[["invoice_no", "description", "quantity", "unitprice", "revenue", "country"]]

display(highest_value_lines.head(10))

# Example: filter sales from the United Kingdom.
uk_sales = sales_df[sales_df["country"] == "United Kingdom"]
print("UK sales revenue: £{:,.2f}".format(uk_sales["revenue"].sum()))

## 16. Write an automatic analysis summary

In [ ]:
best_product = top_products.index[0]
best_country = country_sales.index[0]
best_month = monthly_sales.idxmax()
best_month_revenue = monthly_sales.max()

print("E-COMMERCE SALES ANALYSIS SUMMARY")
print("-" * 45)
print(f"Valid sales lines analyzed: {len(sales_df):,}")
print(f"Total revenue: £{total_revenue:,.2f}")
print(f"Completed orders: {total_orders:,}")
print(f"Identified customers: {identified_customers:,}")
print(f"Top product by revenue: {best_product}")
print(f"Top country by revenue: {best_country}")
print(f"Highest-revenue month: {best_month} (£{best_month_revenue:,.2f})")

print("\nInterpretation:")
print("Use the charts and summary tables to explain sales trends.")
print("Remember: revenue is calculated as quantity multiplied by unit price.")
print("This is historical transaction data, so findings describe this retailer and period only.")

## 17. Export cleaned data and summary tables

In [ ]:
sales_output = "ecommerce_cleaned_sales.csv"
monthly_output = "monthly_sales_summary.csv"
country_output = "country_sales_summary.csv"

# These files are generated when the notebook is run.
sales_df.to_csv(sales_output, index=False)
monthly_summary.to_csv(monthly_output)
country_summary.to_csv(country_output)

print("Exported files:")
print(sales_output)
print(monthly_output)
print(country_output)

# In Google Colab, uncomment these lines if you want to download the outputs.
# from google.colab import files
# files.download(monthly_output)
# files.download(country_output)

## 18. Conclusion

This project explored real-world online retail transactions using Python, NumPy, Pandas and Matplotlib. It inspected data quality, removed duplicate records, handled unusable rows, identified cancellations and invalid sales lines, and calculated sales revenue in GBP.

The analysis grouped and aggregated transactions to examine monthly revenue, top products, country-level sales, customer activity and order patterns. The charts and summary tables help communicate the main findings.

**Limitations:** The dataset represents one UK-based online retailer over a historical period (December 2010 to December 2011). It should not be treated as a description of all modern e-commerce businesses. Missing customer IDs are excluded from customer-level calculations, but can remain in overall sales analysis. Cancellations and non-positive sales lines are excluded from the completed-sales revenue analysis.

## Dataset citation

Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

Dataset page: https://archive.ics.uci.edu/dataset/352/online-retail